# Teaching Loop Experiments (Phases 0–5)

Self-contained runner and analysis

In [1]:
"""Global imports and path setup."""

from pathlib import Path
from typing import Dict, Any, List, Optional
import re
import json
import sys
import random
import yaml
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

project_root = Path("..").resolve()
sys.path.append(str(project_root))
LOG_ROOT = project_root / "logs"
EXPERIMENTS_ROOT = LOG_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_CONFIG_PATH = project_root / "config" / "simplified_config.yml"

print("project_root:", project_root)
print("LOG_ROOT:", LOG_ROOT)
print("EXPERIMENTS_ROOT:", EXPERIMENTS_ROOT)


project_root: C:\Users\ham25\Desktop\Teaching-light-weight-llm-based-project
LOG_ROOT: C:\Users\ham25\Desktop\Teaching-light-weight-llm-based-project\logs
EXPERIMENTS_ROOT: C:\Users\ham25\Desktop\Teaching-light-weight-llm-based-project\logs\experiments


In [2]:
# Path validation to avoid missing-file errors
required_paths = [
    BASE_CONFIG_PATH,
    project_root / 'data' / 'alpaca_20.jsonl',
    project_root / 'data' / 'alpaca_100.jsonl',
    project_root / 'data' / 'medical_all_clean.jsonl',
]
for rp in required_paths:
    if not rp.exists():
        raise FileNotFoundError(f'Missing required path: {rp}')
print('All required paths present.')


All required paths present.


## Helpers (I/O, config, runner)

Memory files (JSONL + FAISS index) are per experiment under `logs/experiments/phaseX/`.
Summary/debug_per_round have a consistent schema (tokens included).

In [3]:
"""JSONL helpers and config utilities."""

import re

def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]", "_", name)


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def flatten_summary_records(records: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    if "config_used" in df.columns:
        cfg = df["config_used"].apply(pd.Series)
        df = pd.concat([df.drop(columns=["config_used"]), cfg], axis=1)
    if "metrics" in df.columns:
        met = df["metrics"].apply(pd.Series)
        df = pd.concat([df.drop(columns=["metrics"]), met], axis=1)
    return df

def get_phase_paths(phase_id: str, experiment_id: Optional[str] = None) -> Dict[str, Path]:
    root = EXPERIMENTS_ROOT / f"phase{phase_id}"
    root.mkdir(parents=True, exist_ok=True)
    paths = {
        "root": root,
        "summary": root / "summary.jsonl",
        "debug_per_round": root / "debug_per_round.jsonl",
    }
    mem_base = root / safe_name(experiment_id if experiment_id else "memory")
    paths["memory_store"] = mem_base.with_suffix(".jsonl")
    paths["memory_index"] = mem_base.with_suffix(".index")
    paths["memory_ids"] = mem_base.with_suffix(".ids.json")
    return paths

def load_base_config() -> Dict[str, Any]:
    with BASE_CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def deep_set(cfg: Dict[str, Any], key_path: str, value: Any) -> None:
    keys = key_path.split(".")
    cur = cfg
    for k in keys[:-1]:
        if k not in cur or not isinstance(cur[k], dict):
            cur[k] = {}
        cur = cur[k]
    cur[keys[-1]] = value

def build_experiment_config(base_cfg: Dict[str, Any], overrides: Dict[str, Any]) -> Dict[str, Any]:
    cfg = json.loads(json.dumps(base_cfg))
    for key_path, value in overrides.items():
        deep_set(cfg, key_path, value)
    return cfg

def save_temp_config(cfg: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)


In [4]:
"""Dataset loader and generic experiment runner."""

def load_dataset(path: Path, max_samples: Optional[int] = None) -> List[Dict[str, Any]]:
    data = load_jsonl(path)
    if max_samples is not None:
        data = data[:max_samples]
    return data

from simplified_teaching_loop import SimplifiedTeachingLoop

def run_single_experiment(
    phase_id: str,
    experiment_id: str,
    questions: List[Dict[str, Any]],
    overrides: Dict[str, Any],
    config_used: Dict[str, Any],
    seed: int = 42,
) -> Dict[str, Any]:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

    exp_id_safe = safe_name(experiment_id)
    paths = get_phase_paths(phase_id, exp_id_safe)
    base_cfg = load_base_config()
    memory_overrides = {
        "memory.storage_path": str(paths["memory_store"]),
        "memory.index_path": str(paths["memory_index"]),
    }
    logging_overrides = {"logging.debug": False, "logging.save_rounds": False, "logging.flat_log_enabled": False}
    merged = {**memory_overrides, **logging_overrides, **overrides}
    exp_cfg = build_experiment_config(base_cfg, merged)
    tmp_cfg_path = paths["root"] / "configs" / f"{exp_id_safe}.yml"
    save_temp_config(exp_cfg, tmp_cfg_path)

    loop = SimplifiedTeachingLoop(config_path=str(tmp_cfg_path))

    total_em = total_rouge = total_sem = 0.0
    total_blind = total_comp = 0.0
    total_rounds = 0
    memory_hits = 0
    num_questions = len(questions)

    for idx, item in enumerate(questions, start=1):
        if idx == 1 or idx % 5 == 0 or idx == num_questions:
            print(f"[phase{phase_id}] {experiment_id}: {idx}/{num_questions}")
        q = item.get("question", "")
        # Ground truth fallback: answer | reference | output
        gt = item.get("answer") or item.get("reference") or item.get("output") or ""
        qid = item.get("id", f"q-{idx}")

        result = loop.run(question=q, ground_truth=gt, question_id=qid, question_idx=idx)
        history = result.get("history", [])
        total_rounds += len(history)

        if history:
            last = history[-1]
            scores = last.get("scores", {})
            total_em += float(scores.get("exact_match", 0.0))
            total_rouge += float(scores.get("rouge_l", 0.0))
            total_sem += float(scores.get("semantic_sim", 0.0))
            total_blind += float(scores.get("blind_score", 0.0))
            total_comp += float(scores.get("comparison_score", 0.0))

        if any(r.get("memory_used") for r in history):
            memory_hits += 1

        for r in history:
            debug_rec = {
                "phase": f"phase{phase_id}",
                "experiment_id": experiment_id,
                "question_id": qid,
                "question_idx": idx,
                "round": r.get("round"),
                "question": q,
                "answer": r.get("answer"),
                "scores": r.get("scores", {}),
                "final_score": r.get("final_score", 0.0),
                "passed": r.get("passed", False),
                "memory_used": r.get("memory_used", False),
                "time_ms": r.get("time_ms", 0),
                "timestamp": r.get("timestamp", datetime.utcnow().isoformat()),
            }
            append_jsonl(paths["debug_per_round"], debug_rec)

    if num_questions > 0:
        avg_em = total_em / num_questions
        avg_rouge = total_rouge / num_questions
        avg_sem = total_sem / num_questions
        avg_blind = total_blind / num_questions
        avg_comp = total_comp / num_questions
        avg_rounds = total_rounds / num_questions
        memory_hit_rate = memory_hits / num_questions
    else:
        avg_em = avg_rouge = avg_sem = avg_blind = avg_comp = avg_rounds = memory_hit_rate = 0.0

    student_tokens_total = getattr(loop.student, "total_tokens", 0)
    teacher_tokens_total = getattr(loop.teacher, "total_tokens", 0)
    student_teacher_tokens = student_tokens_total + teacher_tokens_total

    summary = {
        "experiment_id": experiment_id,
        "phase": f"phase{phase_id}",
        "num_questions": num_questions,
        "seed": seed,
        "metrics": {
            "exact_match": avg_em,
            "rouge_l": avg_rouge,
            "semantic_similarity": avg_sem,
            "blind_judge": avg_blind,
            "comparison_judge": avg_comp,
        },
        "avg_rounds": avg_rounds,
        "memory_hits": memory_hits,
        "memory_hit_rate": memory_hit_rate,
        "student_tokens_total": student_tokens_total,
        "teacher_tokens_total": teacher_tokens_total,
        "student_teacher_tokens": student_teacher_tokens,
        "timestamp": datetime.utcnow().isoformat(),
        "config_used": config_used,
    }

    append_jsonl(paths["summary"], summary)
    return summary

DATA_ALPACA_20 = project_root / "data" / "alpaca_20.jsonl"
DATA_ALPACA_100 = project_root / "data" / "alpaca_100.jsonl"
DATA_MEDICAL_ALL = project_root / "data" / "medical_all_clean.jsonl"
WEIGHTS_TUNED = {"blind_score": 0.2425, "comparison_score": 0.3933, "semantic_sim": 0.2142, "rouge_l": 0.10, "exact_match": 0.05}


c:\Users\ham25\.conda\envs\tlw\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Phase 0 - Baseline 

Single-round baseline without teacher or memory.

In [5]:
phase0_experiments = [
    {
        "experiment_id": "P0-Baseline-Alpaca20",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {"loop.max_rounds": 1, "loop.enable_last_chance": False, "memory.top_k": 0},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "baseline",
            "teacher_feedback_style": None,
            "max_rounds": 1,
            "pass_threshold": None,
            "memory_top_k": 0,
            "memory_similarity_threshold": None,
            "enable_last_chance": False,
            "judge_mode": "hybrid",
        },
    },
]


In [6]:
phase0_summaries = []
for exp in phase0_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="0",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase0_summaries.append(summary)
phase0_summaries


2025-11-28 06:43:56,810 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 06:43:56,810 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 06:43:56,822 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 06:43:56,823 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 06:43:56,839 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 06:43:56,840 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 06:44:00,732 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

[{'experiment_id': 'P0-Baseline-Alpaca20',
  'phase': 'phase0',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.05,
   'rouge_l': 0.46014797241283223,
   'semantic_similarity': 0.7235516019165515,
   'blind_judge': 0.8700000000000003,
   'comparison_judge': 0.8550000000000001},
  'avg_rounds': 1.0,
  'memory_hits': 0,
  'memory_hit_rate': 0.0,
  'student_tokens_total': 3112,
  'teacher_tokens_total': 7343,
  'student_teacher_tokens': 10455,
  'timestamp': '2025-11-27T19:45:06.554355',
  'config_used': {'domain': 'alpaca',
   'student_prompt_strategy': 'baseline',
   'teacher_feedback_style': None,
   'max_rounds': 1,
   'pass_threshold': None,
   'memory_top_k': 0,
   'memory_similarity_threshold': None,
   'enable_last_chance': False,
   'judge_mode': 'hybrid'}}]

## Phase 1 - Prompt and Teacher Strategy

Compare student prompt + teacher feedback styles, including a last-chance variant.

In [7]:
PH1_COMMON = {"loop.max_rounds": 3, "memory.top_k": 3, "memory.similarity_threshold": 0.8356, "loop.enable_last_chance": False}

phase1_experiments = [
    {
        "experiment_id": "P1-Minimal-Template",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "teacher.feedback_style": "template", "teacher.feedback.use_cot": False},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "minimal",
            "teacher_feedback_style": "template",
            "max_rounds": 3,
            "enable_last_chance": False,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
    {
        "experiment_id": "P1-Minimal-CoT",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "teacher.feedback_style": "cot", "teacher.feedback.use_cot": True},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "minimal",
            "teacher_feedback_style": "cot",
            "max_rounds": 3,
            "enable_last_chance": False,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
    {
        "experiment_id": "P1-Minimal-CoT-LastChance",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "teacher.feedback_style": "cot", "teacher.feedback.use_cot": True, "loop.enable_last_chance": True, "loop.ground_truth_hint_round": 99},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "minimal",
            "teacher_feedback_style": "cot",
            "max_rounds": 3,
            "enable_last_chance": True,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
    {
        "experiment_id": "P1-Structured-CoT",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "student.prompt_strategy": "structured", "teacher.feedback_style": "cot", "teacher.feedback.use_cot": True},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "structured",
            "teacher_feedback_style": "cot",
            "max_rounds": 3,
            "enable_last_chance": False,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
    {
        "experiment_id": "P1-Reflective-CoT",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "student.prompt_strategy": "reflective", "teacher.feedback_style": "cot", "teacher.feedback.use_cot": True},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "reflective",
            "teacher_feedback_style": "cot",
            "max_rounds": 3,
            "enable_last_chance": False,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
    {
        "experiment_id": "P1-Minimal-DirectTemplate",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {**PH1_COMMON, "teacher.feedback_style": "direct_template", "teacher.feedback.use_cot": False},
        "config_used": {
            "domain": "alpaca",
            "student_prompt_strategy": "minimal",
            "teacher_feedback_style": "direct_template",
            "max_rounds": 3,
            "enable_last_chance": False,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8356,
            "judge_mode": "hybrid",
        },
    },
]


In [8]:
phase1_summaries = []
for exp in phase1_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="1",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase1_summaries.append(summary)
phase1_summaries


2025-11-28 06:45:06,628 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 06:45:06,628 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 06:45:06,641 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 06:45:06,641 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 06:45:06,659 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 06:45:06,659 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 06:45:10,024 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

[{'experiment_id': 'P1-Minimal-Template',
  'phase': 'phase1',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.05,
   'rouge_l': 0.5234726570943115,
   'semantic_similarity': 0.753419628739357,
   'blind_judge': 0.8425000000000002,
   'comparison_judge': 0.8799999999999999},
  'avg_rounds': 2.4,
  'memory_hits': 0,
  'memory_hit_rate': 0.0,
  'student_tokens_total': 11639,
  'teacher_tokens_total': 14367,
  'student_teacher_tokens': 26006,
  'timestamp': '2025-11-27T19:48:03.354358',
  'config_used': {'domain': 'alpaca',
   'student_prompt_strategy': 'minimal',
   'teacher_feedback_style': 'template',
   'max_rounds': 3,
   'enable_last_chance': False,
   'memory_top_k': 3,
   'memory_similarity_threshold': 0.8356,
   'judge_mode': 'hybrid'}},
 {'experiment_id': 'P1-Minimal-CoT',
  'phase': 'phase1',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.25,
   'rouge_l': 0.7171314559448999,
   'semantic_similarity': 0.8484163984656334,
   'blind_judge'

## Phase 2 - Metric and Threshold Tuning

Explore metric weights and pass_threshold to get a tuned configuration (high score, few rounds).

In [9]:
PH2_COMMON = {"loop.max_rounds": 3, "memory.top_k": 3, "memory.similarity_threshold": 0.8356, "logging.debug": False, "logging.save_rounds": False, "logging.flat_log_enabled": False}

# Grid search (focused)
# pass_threshold: {0.85, 0.90}
# student_temperature: {0.0, 0.2}
# teacher_temperature: {0.2, 0.4}
# memory.similarity_threshold: {0.8356, 0.88}
pass_thresholds = [0.85, 0.90]
student_temps = [0.0, 0.2]
teacher_temps = [0.2, 0.4]
sim_thresholds = [0.75, 0.8, 0.8356]

phase2_experiments = []
for pt in pass_thresholds:
    for st in student_temps:
        for tt in teacher_temps:
            for sim in sim_thresholds:
                sim_tag = str(sim).replace(".", "")[:4]
                exp_id = "P2-PT{:02d}_Ts{}_Tt{}_Sim{}".format(int(pt*100), st, tt, sim_tag)
                phase2_experiments.append({
                    "experiment_id": exp_id,
                    "dataset": DATA_ALPACA_20,
                    "max_questions": 20,
                    "overrides": {**PH2_COMMON, "teacher.pass_threshold": pt, "teacher.metrics.weights": WEIGHTS_TUNED, "student.temperature": st, "teacher.temperature": tt, "memory.similarity_threshold": sim},
                    "config_used": {
                        "domain": "alpaca",
                        "student_prompt_strategy": "minimal",
                        "teacher_feedback_style": "cot",
                        "max_rounds": 3,
                        "pass_threshold": pt,
                        "metric_weights": WEIGHTS_TUNED,
                        "memory_top_k": 3,
                        "memory_similarity_threshold": sim,
                        "student_temperature": st,
                        "teacher_temperature": tt,
                        "judge_mode": "hybrid",
                    },
                })



In [10]:
phase2_summaries = []
for exp in phase2_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="2",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase2_summaries.append(summary)
phase2_summaries


2025-11-28 07:00:06,873 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 07:00:06,874 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 07:00:06,884 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 07:00:06,885 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 07:00:06,900 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 07:00:06,900 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 07:00:10,627 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

[{'experiment_id': 'P2-PT85_Ts0.0_Tt0.2_Sim075',
  'phase': 'phase2',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.2,
   'rouge_l': 0.7118103413670163,
   'semantic_similarity': 0.838678103685379,
   'blind_judge': 0.8500000000000002,
   'comparison_judge': 0.95},
  'avg_rounds': 2.35,
  'memory_hits': 0,
  'memory_hit_rate': 0.0,
  'student_tokens_total': 12161,
  'teacher_tokens_total': 19809,
  'student_teacher_tokens': 31970,
  'timestamp': '2025-11-27T20:03:00.326543',
  'config_used': {'domain': 'alpaca',
   'student_prompt_strategy': 'minimal',
   'teacher_feedback_style': 'cot',
   'max_rounds': 3,
   'pass_threshold': 0.85,
   'metric_weights': {'blind_score': 0.2425,
    'comparison_score': 0.3933,
    'semantic_sim': 0.2142,
    'rouge_l': 0.1,
    'exact_match': 0.05},
   'memory_top_k': 3,
   'memory_similarity_threshold': 0.75,
   'student_temperature': 0.0,
   'teacher_temperature': 0.2,
   'judge_mode': 'hybrid'}},
 {'experiment_id': 'P2-PT85_Ts0.

## Phase 3 - Judge Strategy (blind vs comparison vs deterministic)

Evaluate judging schemes on Alpaca-20 using the tuned weights:
- Hybrid (blind + comparison) ? reference.
- Comparison-only (blind weight = 0).
- Blind-only (comparison weight = 0).
Pass threshold fixed from Phase 2 tuning.

In [11]:
phase3_experiments = [
    {
        "experiment_id": "P3-Hybrid",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.80,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "judge_mode": "hybrid",
            "weights": WEIGHTS_TUNED,
            "pass_threshold": 0.90,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.80,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
            "max_rounds": 3,
        },
    },
    {
        "experiment_id": "P3-ComparisonOnly",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.80,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": {
                "blind_score": 0.0,
                "comparison_score": 0.55,
                "semantic_sim": 0.25,
                "rouge_l": 0.15,
                "exact_match": 0.05,
            },
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "judge_mode": "comparison_only",
            "weights": {
                "blind_score": 0.0,
                "comparison_score": 0.55,
                "semantic_sim": 0.25,
                "rouge_l": 0.15,
                "exact_match": 0.05,
            },
            "pass_threshold": 0.90,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.80,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
            "max_rounds": 3,
        },
    },
    {
        "experiment_id": "P3-BlindOnly",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.80,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": {
                "blind_score": 0.60,
                "comparison_score": 0.0,
                "semantic_sim": 0.25,
                "rouge_l": 0.10,
                "exact_match": 0.05,
            },
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "judge_mode": "blind_only",
            "weights": {
                "blind_score": 0.60,
                "comparison_score": 0.0,
                "semantic_sim": 0.25,
                "rouge_l": 0.10,
                "exact_match": 0.05,
            },
            "pass_threshold": 0.90,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.80,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
            "max_rounds": 3,
        },
    },
]


In [12]:
phase3_summaries = []
for exp in phase3_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="3",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase3_summaries.append(summary)
phase3_summaries


2025-11-28 12:58:09,760 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 12:58:09,761 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 12:58:09,771 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 12:58:09,771 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 12:58:09,782 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 12:58:09,782 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 12:58:14,359 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

[{'experiment_id': 'P3-Hybrid',
  'phase': 'phase3',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.35,
   'rouge_l': 0.7326636682271619,
   'semantic_similarity': 0.8710941478610039,
   'blind_judge': 0.8450000000000001,
   'comparison_judge': 0.93},
  'avg_rounds': 2.65,
  'memory_hits': 0,
  'memory_hit_rate': 0.0,
  'student_tokens_total': 14250,
  'teacher_tokens_total': 24727,
  'student_teacher_tokens': 38977,
  'timestamp': '2025-11-28T02:01:10.568790',
  'config_used': {'domain': 'alpaca',
   'judge_mode': 'hybrid',
   'weights': {'blind_score': 0.2425,
    'comparison_score': 0.3933,
    'semantic_sim': 0.2142,
    'rouge_l': 0.1,
    'exact_match': 0.05},
   'pass_threshold': 0.9,
   'memory_top_k': 3,
   'memory_similarity_threshold': 0.8,
   'student_temperature': 0.2,
   'teacher_temperature': 0.4,
   'max_rounds': 3}},
 {'experiment_id': 'P3-ComparisonOnly',
  'phase': 'phase3',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.35,


## Phase 4 - Domain + Memory Ablation (Alpaca vs Medical)

Test tuned config on both domains with memory ON/OFF (top_k=3 vs 0) to assess memory impact and domain gap.

In [13]:
phase4_experiments = [
    {
        "experiment_id": "P4-Alpaca-MemON",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.80,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.80,
            "max_rounds": 3,
            "pass_threshold": 0.90,
            "metric_weights": WEIGHTS_TUNED,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P4-Alpaca-MemOFF",
        "dataset": DATA_ALPACA_20,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "memory_top_k": 0,
            "memory_similarity_threshold": 0.99,
            "max_rounds": 3,
            "pass_threshold": 0.90,
            "metric_weights": WEIGHTS_TUNED,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P4-Medical-MemON",
        "dataset": DATA_MEDICAL_ALL,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.80,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "medical",
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.80,
            "max_rounds": 3,
            "pass_threshold": 0.90,
            "metric_weights": WEIGHTS_TUNED,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P4-Medical-MemOFF",
        "dataset": DATA_MEDICAL_ALL,
        "max_questions": 20,
        "overrides": {
            "loop.max_rounds": 3,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.90,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "medical",
            "memory_top_k": 0,
            "memory_similarity_threshold": 0.99,
            "max_rounds": 3,
            "pass_threshold": 0.90,
            "metric_weights": WEIGHTS_TUNED,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
]


In [14]:
phase4_summaries = []
for exp in phase4_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="4",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase4_summaries.append(summary)
phase4_summaries


2025-11-28 13:07:10,901 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 13:07:10,901 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 13:07:10,917 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 13:07:10,917 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 13:07:10,934 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 13:07:10,934 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 13:07:14,532 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

[{'experiment_id': 'P4-Alpaca-MemON',
  'phase': 'phase4',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.25,
   'rouge_l': 0.7182246079910113,
   'semantic_similarity': 0.860859851539135,
   'blind_judge': 0.8275000000000002,
   'comparison_judge': 0.89},
  'avg_rounds': 2.7,
  'memory_hits': 0,
  'memory_hit_rate': 0.0,
  'student_tokens_total': 14046,
  'teacher_tokens_total': 25874,
  'student_teacher_tokens': 39920,
  'timestamp': '2025-11-28T02:10:10.256697',
  'config_used': {'domain': 'alpaca',
   'memory_top_k': 3,
   'memory_similarity_threshold': 0.8,
   'max_rounds': 3,
   'pass_threshold': 0.9,
   'metric_weights': {'blind_score': 0.2425,
    'comparison_score': 0.3933,
    'semantic_sim': 0.2142,
    'rouge_l': 0.1,
    'exact_match': 0.05},
   'student_temperature': 0.2,
   'teacher_temperature': 0.4}},
 {'experiment_id': 'P4-Alpaca-MemOFF',
  'phase': 'phase4',
  'num_questions': 20,
  'seed': 42,
  'metrics': {'exact_match': 0.3,
   'rouge_l': 0.73

## Phase 5 - Full Proof-of-Concept

Six experiments: baseline vs tuned (mem ON/OFF) on Alpaca-100 and Medical-100.
- Baseline: student-only, no memory, single round.
- Tuned: teacher + feedback + memory, max_rounds=10, tuned metrics/threshold.

In [15]:
phase5_experiments = [
    {
        "experiment_id": "P5-Baseline-Alpaca-100",
        "dataset": DATA_ALPACA_100,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 1,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.0,
            "teacher.metrics.weights": {
                "blind_score": 0.0,
                "comparison_score": 0.0,
                "semantic_sim": 0.0,
                "rouge_l": 0.0,
                "exact_match": 1.0,
            },
            "teacher.enabled": False,
        },
        "config_used": {
            "domain": "alpaca",
            "mode": "baseline",
            "max_rounds": 1,
            "memory_top_k": 0,
            "teacher_enabled": False,
        },
    },
    {
        "experiment_id": "P5-Baseline-Medical-100",
        "dataset": DATA_MEDICAL_ALL,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 1,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.0,
            "teacher.metrics.weights": {
                "blind_score": 0.0,
                "comparison_score": 0.0,
                "semantic_sim": 0.0,
                "rouge_l": 0.0,
                "exact_match": 1.0,
            },
            "teacher.enabled": False,
        },
        "config_used": {
            "domain": "medical",
            "mode": "baseline",
            "max_rounds": 1,
            "memory_top_k": 0,
            "teacher_enabled": False,
        },
    },
    {
        "experiment_id": "P5-Tuned-MemON-Alpaca-100",
        "dataset": DATA_ALPACA_100,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 10,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.8,
            "teacher.pass_threshold": 0.9,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "mode": "tuned_mem_on",
            "max_rounds": 10,
            "pass_threshold": 0.9,
            "metric_weights": WEIGHTS_TUNED,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P5-Tuned-MemOFF-Alpaca-100",
        "dataset": DATA_ALPACA_100,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 10,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.9,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "alpaca",
            "mode": "tuned_mem_off",
            "max_rounds": 10,
            "pass_threshold": 0.9,
            "metric_weights": WEIGHTS_TUNED,
            "memory_top_k": 0,
            "memory_similarity_threshold": 0.99,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P5-Tuned-MemON-Medical-100",
        "dataset": DATA_MEDICAL_ALL,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 10,
            "memory.top_k": 3,
            "memory.similarity_threshold": 0.8,
            "teacher.pass_threshold": 0.9,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "medical",
            "mode": "tuned_mem_on",
            "max_rounds": 10,
            "pass_threshold": 0.9,
            "metric_weights": WEIGHTS_TUNED,
            "memory_top_k": 3,
            "memory_similarity_threshold": 0.8,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
    {
        "experiment_id": "P5-Tuned-MemOFF-Medical-100",
        "dataset": DATA_MEDICAL_ALL,
        "max_questions": 100,
        "overrides": {
            "loop.max_rounds": 10,
            "memory.top_k": 0,
            "memory.similarity_threshold": 0.99,
            "teacher.pass_threshold": 0.9,
            "teacher.metrics.weights": WEIGHTS_TUNED,
            "student.temperature": 0.2,
            "teacher.temperature": 0.4,
        },
        "config_used": {
            "domain": "medical",
            "mode": "tuned_mem_off",
            "max_rounds": 10,
            "pass_threshold": 0.9,
            "metric_weights": WEIGHTS_TUNED,
            "memory_top_k": 0,
            "memory_similarity_threshold": 0.99,
            "student_temperature": 0.2,
            "teacher_temperature": 0.4,
        },
    },
]


In [16]:
phase5_summaries = []
for exp in phase5_experiments:
    ds = load_dataset(exp["dataset"], max_samples=exp.get("max_questions"))
    summary = run_single_experiment(
        phase_id="5",
        experiment_id=exp["experiment_id"],
        questions=ds,
        overrides=exp["overrides"],
        config_used=exp["config_used"],
        seed=42,
    )
    phase5_summaries.append(summary)
phase5_summaries


2025-11-28 13:21:17,016 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 13:21:17,016 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 13:21:17,032 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=6000, RPD=14400, interval=2.00s
2025-11-28 13:21:17,032 | INFO     | provider.groq                       | Initialized llama-3.1-8b-instant with limits: RPM=30, TPM=6000, RPD=14400
2025-11-28 13:21:17,045 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000, RPD=1000, interval=2.00s
2025-11-28 13:21:17,046 | INFO     | provider.groq                       | Initialized llama-3.3-70b-versatile with limits: RPM=30, TPM=12000, RPD=1000
2025-11-28 13:21:20,641 | INFO     | provider.ratelimit                  | RateLimiter initialized: RPM=30, TPM=12000,

KeyboardInterrupt: 

## Analysis - Load summaries and build tables/plots

Loads summary.jsonl from phases 0-5, flattens metrics, and produces core visualizations:
- Main results table per phase.
- Phase 5 multi-metric overview (EM, semantic, comparison).
- Memory hit rates (phase 4 & 5 tuned configs).
- Cost vs quality (tokens per question vs EM).
- Per-round dynamics using debug_per_round (phase 5).


In [ ]:
import math

def load_phase_summary(phase_id: str) -> pd.DataFrame:
    path = get_phase_paths(phase_id)["summary"]
    records = load_jsonl(path)
    return flatten_summary_records(records)

df_p0 = load_phase_summary("0")
df_p1 = load_phase_summary("1")
df_p2 = load_phase_summary("2")
df_p3 = load_phase_summary("3")
df_p4 = load_phase_summary("4")
df_p5 = load_phase_summary("5")

df_all = pd.concat([df_p0, df_p1, df_p2, df_p3, df_p4, df_p5], ignore_index=True)
df_all


In [ ]:
# Phase 5 multi-metric overview (EM, semantic, comparison) with CI

df_p5_plot = df_p5.copy()
df_p5_plot['exp_label'] = df_p5_plot['experiment_id'].replace({
    'P5-Baseline-Alpaca-100': 'A-Base',
    'P5-Baseline-Medical-100': 'M-Base',
    'P5-Tuned-MemON-Alpaca-100': 'A-MemON',
    'P5-Tuned-MemOFF-Alpaca-100': 'A-MemOFF',
    'P5-Tuned-MemON-Medical-100': 'M-MemON',
    'P5-Tuned-MemOFF-Medical-100': 'M-MemOFF',
})
mode_colors = {'baseline': '#9ecae1', 'tuned_mem_on': '#31a354', 'tuned_mem_off': '#756bb1'}
df_p5_plot['mode'] = df_p5_plot.get('mode', None)
df_p5_plot['mode'] = df_p5_plot['mode'].fillna(df_p5_plot['experiment_id'].apply(
    lambda x: 'baseline' if 'Baseline' in x else ('tuned_mem_on' if 'MemON' in x else 'tuned_mem_off')
))
colors = df_p5_plot['mode'].map(mode_colors)
x = range(len(df_p5_plot))

# Binomial 95% CI for EM
def em_ci(p, n):
    se = (p * (1 - p) / n) ** 0.5 if n > 0 else 0.0
    return 1.96 * se
em_err = [em_ci(v, int(nq)) for v, nq in zip(df_p5_plot['exact_match'], df_p5_plot['num_questions'])]

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)
metrics = [
    ('exact_match', 'Exact Match', (0.0, 1.0), em_err),
    ('semantic_similarity', 'Semantic Similarity', (0.7, 1.0), None),
    ('comparison_judge', 'Comparison Judge', (0.7, 1.0), None),
]
for ax, (col, title, ylim, err) in zip(axes, metrics):
    ax.bar(x, df_p5_plot[col], color=colors, edgecolor='black', linewidth=0.5, yerr=err, capsize=3)
    ax.set_ylabel(title)
    ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3)
axes[-1].set_xticks(list(x))
axes[-1].set_xticklabels(list(df_p5_plot['exp_label']), rotation=20)
legend_handles = [
    Patch(facecolor=mode_colors['baseline'], label='Baseline'),
    Patch(facecolor=mode_colors['tuned_mem_on'], label='Tuned + MemON'),
    Patch(facecolor=mode_colors['tuned_mem_off'], label='Tuned + MemOFF'),
]
axes[0].legend(handles=legend_handles, title='Mode', loc='upper right')
fig.suptitle('Phase 5 - Multi-metric Overview')
plt.tight_layout()
plt.show()


In [ ]:
# Memory hit rate for Phase 4 and Phase 5 tuned configs

df_mem = pd.concat([df_p4, df_p5], ignore_index=True)
df_mem = df_mem[df_mem.get('memory_top_k', 0).notna()]
plt.figure(figsize=(8, 4))
x = range(len(df_mem))
bars = plt.bar(x, df_mem['memory_hit_rate'] * 100, color='#6baed6', edgecolor='black')
plt.xticks(x, list(df_mem['experiment_id']), rotation=25)
plt.ylabel('Memory Hit Rate (%)')
plt.title('Memory Hit Rate - Phase 4 & 5')
plt.grid(axis='y', alpha=0.3)
for b, v in zip(bars, df_mem['memory_hit_rate'] * 100):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Phase 5 - Cost vs Quality (EM vs Tokens per Question)

fig, ax1 = plt.subplots(figsize=(9, 5))
x = range(len(df_p5_plot))
colors = df_p5_plot['mode'].map(mode_colors)
bars = ax1.bar(x, df_p5_plot['exact_match'], color=colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Exact Match')
ax1.set_ylim(0.0, 1.0)
ax1.grid(axis='y', alpha=0.3)
ax2 = ax1.twinx()
tokens_per_q = df_p5_plot['student_teacher_tokens'] / df_p5_plot['num_questions']
ax2.plot(x, tokens_per_q, color='black', marker='o', linewidth=1.5, label='Tokens per Question')
ax2.set_ylabel('Tokens per Question (S+T)')
ax1.set_xticks(list(x))
ax1.set_xticklabels(list(df_p5_plot['exp_label']), rotation=20)
legend_mode = [
    Patch(facecolor=mode_colors['baseline'], label='Baseline'),
    Patch(facecolor=mode_colors['tuned_mem_on'], label='Tuned + MemON'),
    Patch(facecolor=mode_colors['tuned_mem_off'], label='Tuned + MemOFF'),
]
line_legend = Line2D([0], [0], color='black', marker='o', label='Tokens per Question')
ax1.legend(handles=legend_mode, title='Mode', loc='upper left')
ax2.legend(handles=[line_legend], loc='upper right')
plt.title('Phase 5 - Cost vs Quality')
plt.tight_layout()
plt.show()


In [ ]:
# Per-round dynamics (average final_score) - Phase 5 debug_per_round

debug_p5 = load_jsonl(get_phase_paths("5")["debug_per_round"])
df_debug = pd.DataFrame(debug_p5)
if not df_debug.empty:
    grp = df_debug.groupby(["experiment_id", "round"])  # average per round per experiment
    df_round = grp["final_score"].mean().reset_index()

    plt.figure(figsize=(9, 5))
    for exp_id, sub in df_round.groupby("experiment_id"):
        plt.plot(sub["round"], sub["final_score"], marker="o", label=exp_id)
    plt.xlabel("Round")
    plt.ylabel("Average Final Score")
    plt.title("Phase 5 ? Feedback Loop Dynamics")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No phase 5 debug_per_round data available.")


In [ ]:
# Phase 5 main results table
df_p5_table = df_p5.copy()
df_p5_table['tokens_per_question'] = df_p5_table['student_teacher_tokens'] / df_p5_table['num_questions']
display(df_p5_table[[
    'experiment_id','domain','mode','num_questions',
    'exact_match','rouge_l','semantic_similarity','blind_judge','comparison_judge',
    'avg_rounds','memory_hit_rate','student_tokens_total','teacher_tokens_total','student_teacher_tokens','tokens_per_question'
]].round(3))


In [ ]:
# Phase 5 EM delta vs baseline per domain
baseline = df_p5.set_index('experiment_id')
def get_em(eid): return baseline.loc[eid, 'exact_match'] if eid in baseline.index else None
rows=[]
for domain, base_id, off_id, on_id in [
    ('alpaca','P5-Baseline-Alpaca-100','P5-Tuned-MemOFF-Alpaca-100','P5-Tuned-MemON-Alpaca-100'),
    ('medical','P5-Baseline-Medical-100','P5-Tuned-MemOFF-Medical-100','P5-Tuned-MemON-Medical-100'),
]:
    b = get_em(base_id); off = get_em(off_id); on = get_em(on_id)
    rows.append({'domain':domain,'config':'mem_off','em':off,'delta_vs_base': off - b if b is not None else None})
    rows.append({'domain':domain,'config':'mem_on','em':on,'delta_vs_base': on - b if b is not None else None})
df_em_delta = pd.DataFrame(rows)
display(df_em_delta)
plt.figure(figsize=(6,4))
for i, domain in enumerate(df_em_delta['domain'].unique()):
    sub = df_em_delta[df_em_delta['domain']==domain]
    plt.bar([i*2, i*2+1], sub['delta_vs_base'], color=['#756bb1','#31a354'], edgecolor='black')
plt.xticks([0,1,2,3], ['A-mem_off','A-mem_on','M-mem_off','M-mem_on'], rotation=0)
plt.ylabel('EM Delta vs Baseline')
plt.title('Phase 5 ? Improvement vs Baseline')
plt.axhline(0, color='black', linewidth=0.8)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of comparison_judge per question (baseline vs tuned)
debug_p5 = load_jsonl(get_phase_paths('5')['debug_per_round'])
df_dbg = pd.DataFrame(debug_p5)
if not df_dbg.empty:
    # keep last round per question
    last_round = df_dbg.sort_values(['experiment_id','question_id','round']).groupby(['experiment_id','question_id']).tail(1)
    plt.figure(figsize=(9,5))
    keep_ids = [
        'P5-Baseline-Alpaca-100','P5-Tuned-MemON-Alpaca-100','P5-Tuned-MemOFF-Alpaca-100',
        'P5-Baseline-Medical-100','P5-Tuned-MemON-Medical-100','P5-Tuned-MemOFF-Medical-100'
    ]
    sub = last_round[last_round['experiment_id'].isin(keep_ids)]
    sub.boxplot(column='final_score', by='experiment_id', grid=False, rot=25)
    plt.suptitle('')
    plt.title('Phase 5 ? Final Score Distribution')
    plt.ylabel('Final Score')
    plt.tight_layout()
    plt.show()
else:
    print('No phase 5 debug data for distribution plot.')


In [ ]:
# Helper: ensure numeric columns for analysis to avoid operator issues
def cast_numeric(df: pd.DataFrame, cols):
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    return df


In [ ]:
# Example: before computing deltas, cast numeric
# num_cols = ['exact_match','rouge_l','semantic_similarity','blind_judge','comparison_judge',
#             'avg_rounds','memory_hit_rate','student_teacher_tokens','tokens_per_q']
# df = cast_numeric(df, num_cols)
# Then do: df['em_delta'] = df['em_tuned'] - df['em_base']
# If checking notna on scalars, use pd.notna(x) instead of x.notna()
